# Week 5 Exercise — Emmanuel

Load medical transcription data from **mtsamples.csv** in the `data/` directory (columns: `description`, `medical_specialty`, `sample_name`, `transcription`, `keywords`).

In [ ]:
import os
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import numpy as np
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from litellm import completion
from pydantic import BaseModel, Field
from openai import OpenAI
import gradio as gr




In [ ]:
CSV_PATH = "data/mtsamples.csv"
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
openai = OpenAI()

In [ ]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Expected in the current directory")

df = pd.read_csv(CSV_PATH, index_col=0)
print("Loaded:", CSV_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df

In [ ]:
# Store the entire CSV file as a string (same as on disk: header + rows)
with open(CSV_PATH, encoding="utf-8") as f:
    file_str = f.read()

# Count tokens using tiktoken (OpenAI-style tokenizer)
import tiktoken
encoding = tiktoken.encoding_for_model(MODEL)
num_tokens = len(encoding.encode(file_str))

print("File length (characters):", len(file_str))
print("Number of tokens:", num_tokens)

In [ ]:
# Preview first rows (transcription truncated for display)
df['sample_name'].unique()

In [ ]:

def df_to_documents(df, content_columns=None, metadata_columns=None):
    """Load LangChain Documents from a DataFrame instead of a directory."""
    if content_columns is None:
        content_columns = ["description", "medical_specialty", "sample_name", "transcription", "keywords"]
    if metadata_columns is None:
        metadata_columns = ["medical_specialty", "sample_name", "keywords"]

    documents = []
    for i, row in df.iterrows():
        parts = [f"{col}: {row[col]}" for col in content_columns if pd.notna(row.get(col))]
        page_content = "\n\n".join(parts)
        metadata = {col: str(row[col]) for col in metadata_columns if pd.notna(row.get(col))}
        metadata["row_id"] = i
        documents.append(Document(page_content=page_content, metadata=metadata))
    return documents

In [ ]:
documents = df_to_documents(df)

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print("First chunk (preview):\n", chunks[0].page_content[:400], "...")

In [ ]:

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
batch = collection.get(limit=20, offset=0)
print(batch['metadatas'])

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
retriever.invoke("what is hypertension?")

In [ ]:
llm.invoke("what is hypertension?")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly mdedical consultant.
You are helpful in providing answers to every medical question you have encountered based on your experience.
If and only if the provided context is a relevant medical context, use it to answer the given question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content, docs

In [ ]:
def format_context(docs):
    """Format retrieved chunks as HTML for the context panel (like week5 app.py)."""
    if not docs:
        return "*No context retrieved.*"
    lines = ["<h3>Relevant Context</h3>"]
    for d in docs:
        src = d.metadata.get("medical_specialty", "") or d.metadata.get("sample_name", "—")
        lines.append(f"<p><strong>Source:</strong> {src}</p>")
        lines.append(f"<pre style='white-space: pre-wrap;'>{d.page_content[:2000]}</pre>")
    return "\n".join(lines)


def chat(history):
    """Append assistant reply and return updated history and context HTML."""
    if not history:
        return history, "*No message.*"
    last = history[-1]
    if last.get("role") != "user":
        return history, "*Send a message first.*"
    user_msg = last["content"]
    prior = history[:-1]
    answer, context_docs = answer_question(user_msg, prior)
    history = history + [{"role": "assistant", "content": answer}]
    return history, format_context(context_docs)


def main():
    def submit(message, history):
        history = history or []
        if not message or not message.strip():
            return "", history
        return "", history + [{"role": "user", "content": message.strip()}]

    with gr.Blocks(title="Medical Transcription RAG", theme=gr.themes.Soft()) as ui:
        gr.Markdown("# Medical Transcription Assistant\nAsk questions about medical transcriptions from the mtsamples dataset.")
        with gr.Row():
            with gr.Column(scale=1):
                chatbot = gr.Chatbot(label="Conversation", height=500, type="messages", show_copy_button=True)
                msg = gr.Textbox(placeholder="Ask anything about medical transcriptions...", show_label=False)
            with gr.Column(scale=1):
                context_md = gr.Markdown(value="*Retrieved context will appear here.*", elem_id="context", height=500)
        msg.submit(submit, inputs=[msg, chatbot], outputs=[msg, chatbot]).then(
            chat, inputs=chatbot, outputs=[chatbot, context_md]
        )
    ui.launch(inbrowser=True)


main()

In [ ]:
answer, docs = answer_question("When is mild tricuspid regurgitation", [])
answer

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10
embedding_model = "text-embedding-3-large"

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

In [ ]:
question = "What is an example of PREOPERATIVE DIAGNOSIS"
chunks = fetch_context_unranked(question)

In [ ]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
question = "What comes to mind when you hear bilateral vasectomy?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "vasectomy" in c.page_content.lower():
        print(index)

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly mdedical consultant.
You are helpful in providing answers to every medical question you have encountered based on your experience.
If and only if the provided context is a relevant medical context, use it to answer the given question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['medical_specialty']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("What happened with the umbilical hernia", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("What happened with the umbilical hernia", [])